# mse-reconstruction-loss — ex2: per-pixel saliency map from per-element MSE

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `mse-reconstruction-loss`. Running the final beacon cell reports progress against the `Generative: MSE reconstruction loss` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Generative: MSE reconstruction loss` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`mse-reconstruction-loss`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "mse-reconstruction-loss"
DD_SUBTOPIC = "Generative: MSE reconstruction loss"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## MSE reconstruction loss — deepening refresher

`F.mse_loss(decoded, original)` has three reductions:
- `'mean'` (default) → scalar averaged over **every** element
- `'sum'` → scalar summed over every element
- `'none'` → per-element tensor with the original shape, you choose how to collapse it

**Per-axis averaging.** Once you have `(B, 1, H, W)` per-element error, you can average over any subset of axes. Averaging over `[0, 1]` gives a `(H, W)` **saliency map** — the average per-pixel reconstruction error across the whole batch. Hot spots reveal pixels the autoencoder struggles with regardless of which sample it sees (often the edges of digits, or the corners that are usually zero).

**Why this is NOT what you backprop.** The saliency map is for DIAGNOSIS — the scalar `.mean()` loss is what `.backward()` consumes. Backpropagating a `(H, W)` tensor implicitly sums it, which is the wrong reduction (training would be `H*W` times stronger).

### Exercise 2 — per-pixel saliency map from per-element MSE

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Analyze
> LO: Analyze a batch of MNIST reconstructions by computing the (H, W) average per-pixel MSE across the batch and channel axes, identifying where the autoencoder fails most.
> Keywords: mse, reduction-none, saliency, diagnosis
> ```

**KCs targeted:** `mse-reduction-none-shape`, `mean-over-batch-and-channel-axes`

Implement `ex2_pixel_saliency(original, decoded)`. Build the average per-pixel reconstruction error across a whole batch — a diagnostic heatmap that tells you which pixels the autoencoder gets wrong regardless of which image it sees.

1. `original` and `decoded` both have shape `(B, 1, H, W)`.
2. Call `F.mse_loss(decoded, original, reduction='none')` — result has shape `(B, 1, H, W)` (per-element squared error).
3. Average over the batch axis (`dim=0`) AND the channel axis (`dim=1`) to collapse to `(H, W)`. Use a single `.mean(dim=[0, 1])` call.
4. Return the `(H, W)` saliency map.

**Do NOT** call `F.mse_loss(..., reduction='mean')` first and then try to recover per-pixel detail — once you've reduced to a scalar, the per-pixel info is gone.

**Do NOT** use a Python loop over the batch. The whole point is a single tensor reduction.

Inputs: `(B, 1, H, W)` float tensors.
Output: `(H, W)` float tensor — the average per-pixel error.

In [ ]:
def ex2_pixel_saliency(original: Tensor, decoded: Tensor) -> Tensor:
    import torch.nn.functional as F
    per_elem = F.mse_loss(decoded, original, reduction='none')   # (B, 1, H, W)
    return per_elem.mean(dim=[0, 1])                              # (H, W)


<details><summary>Solution</summary>

```python
def ex2_pixel_saliency(original: Tensor, decoded: Tensor) -> Tensor:
    import torch.nn.functional as F
    per_elem = F.mse_loss(decoded, original, reduction='none')   # (B, 1, H, W)
    return per_elem.mean(dim=[0, 1])                              # (H, W)
```

**Why `reduction='none'`.** It's the only reduction that preserves the `(B, 1, H, W)` shape, leaving you free to reduce over a chosen subset of axes. `'mean'` and `'sum'` both collapse to a scalar — you can't recover per-pixel information after that.

**Why `dim=[0, 1]` and not `dim=0` then `dim=0` again.** Calling `mean(dim=0)` twice in a row would average over batch (`dim=0` of `(B,1,H,W)`) and then again over the channel axis (now `dim=0` of `(1,H,W)`). Functionally equivalent here because the channel size is 1, but `dim=[0, 1]` is the explicit single-call form that scales to multi-channel images (e.g. `(B, 3, H, W)` for RGB).

**Saliency vs training loss.** This is a DIAGNOSTIC. The scalar `F.mse_loss(decoded, original)` is what `.backward()` consumes — backpropagating a `(H, W)` map would implicitly sum it and apply `H*W`-times-too-strong gradients.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()